# Rebound-state factor target contract v1

This notebook turns the upstream `rebound_state_factor_mining_v1` signal into a fold-local target-contract candidate. It keeps the scope narrow: no model is trained here. The output is a target revision contract and a label/target stability audit for the next BC or PPO replay gate.

Design notes:
- Source signal: `spread_confirmed_pullback__mom5_leq0.20_ratio21_geq0.75` from the factor card.
- Source target table: `state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv`.
- Candidate variants compare rebound-only target lifts against release-style risk target lifts.
- Selection requires no eval-window leakage, zero duplicate-date label conflict, stable state-head target lifts, and effective target lift on every factor-matched target row.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    root = start.resolve()
    while root != root.parent:
        if (root / 'src' / 'soxl_quant').exists() and (root / 'runtime').exists():
            return root
        root = root.parent
    raise RuntimeError('repo root not found')

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.representation_diagnostics import (
    StateHeadConsensusTargetConfig,
    derive_state_head_consensus_targets,
    score_state_head_lift_stability,
)
from src.soxl_quant.app.stable_label_discovery import (
    BinaryLabelDiscoveryConfig,
    score_binary_label_candidates,
)
from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    summarize_state_objective_target_lift_audit,
)

EXPERIMENT = 'rebound_state_factor_target_contract_v1'
CARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'state_objective_fold_native_teacher_router_v1' / 'fold_native_teacher_targets.csv'
FACTOR_CARD = ROOT / 'runtime' / 'factor' / 'card' / 'rebound_state_factor_mining_v1'
FACTOR_ROWS_PATH = FACTOR_CARD / 'selected_factor_rows.csv'
FACTOR_SUMMARY_PATH = FACTOR_CARD / 'summary.json'
HANDOFF_SUMMARY_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'rebound_state_factor_target_contract_handoff_v1' / 'summary.json'

targets = pd.read_csv(TARGET_PATH, parse_dates=['datetime'])
factor_rows = pd.read_csv(FACTOR_ROWS_PATH, parse_dates=['datetime'])
factor_payload = json.loads(FACTOR_SUMMARY_PATH.read_text(encoding='utf-8'))
factor_summary = factor_payload.get('summary', factor_payload)
handoff_payload = json.loads(HANDOFF_SUMMARY_PATH.read_text(encoding='utf-8')) if HANDOFF_SUMMARY_PATH.exists() else {}
handoff_summary = handoff_payload.get('summary', handoff_payload)

for column in ['train_start', 'train_end', 'eval_start', 'eval_end']:
    if column in targets.columns:
        targets[column] = pd.to_datetime(targets[column], errors='coerce')
targets['date_key'] = pd.to_datetime(targets['datetime']).dt.normalize()
factor_rows['date_key'] = pd.to_datetime(factor_rows['datetime']).dt.normalize()
selected_dates = set(factor_rows['date_key'].dropna())

numeric_cols = ['teacher_position', 'rebound_target_position', 'risk_target_position', 'objective_weight', 'price']
for column in numeric_cols:
    if column in targets.columns:
        targets[column] = pd.to_numeric(targets[column], errors='coerce')
for column in ['is_rebound_cluster', 'is_downtrend_state', 'rebound_state', 'risk_state']:
    if column in targets.columns:
        targets[column] = targets[column].fillna(False).astype(bool)

price_by_date = (
    targets.dropna(subset=['price'])
    .drop_duplicates('date_key')
    .sort_values('date_key')
    .set_index('date_key')['price']
    .astype(float)
)
for horizon in (5, 10, 21):
    forward = price_by_date.shift(-horizon) / price_by_date - 1.0
    targets[f'forward_return_{horizon}'] = targets['date_key'].map(forward)

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    return value

def current_position(frame: pd.DataFrame) -> pd.Series:
    return pd.Series(
        np.where(
            frame['is_downtrend_state'].astype(bool),
            frame['risk_target_position'],
            np.where(frame['is_rebound_cluster'].astype(bool), frame['rebound_target_position'], frame['teacher_position']),
        ),
        index=frame.index,
        dtype=float,
    )

def make_variant(floor: float, release_risk_target: bool) -> pd.DataFrame:
    result = targets.copy()
    for column in ['teacher_position', 'rebound_target_position', 'risk_target_position']:
        result[column] = pd.to_numeric(result[column], errors='coerce').fillna(0.0).clip(0.0, 1.0)
    for column in ['is_rebound_cluster', 'is_downtrend_state']:
        result[column] = result[column].fillna(False).astype(bool)
    if 'risk_target_position_original' not in result.columns:
        result['risk_target_position_original'] = result['risk_target_position']
    result['risk_target_position_original'] = pd.to_numeric(
        result['risk_target_position_original'], errors='coerce'
    ).fillna(result['risk_target_position']).clip(0.0, 1.0)
    if 'release_gate_state' not in result.columns:
        result['release_gate_state'] = False
    result['release_gate_state'] = result['release_gate_state'].fillna(False).astype(bool)
    result['factor_rebound_state'] = result['date_key'].isin(selected_dates)
    result['factor_target_floor'] = float(floor)
    result['factor_release_risk_target'] = bool(release_risk_target)
    result['target_semantics_revision'] = ''

    mask = result['factor_rebound_state']
    floor_position = np.maximum(result.loc[mask, 'teacher_position'].astype(float), float(floor))
    result.loc[mask, 'is_rebound_cluster'] = True
    if 'rebound_state' in result.columns:
        result.loc[mask, 'rebound_state'] = True
    result.loc[mask, 'rebound_target_position'] = np.maximum(
        result.loc[mask, 'rebound_target_position'].astype(float), floor_position
    )
    if release_risk_target:
        result.loc[mask, 'risk_target_position'] = np.maximum(
            result.loc[mask, 'risk_target_position'].astype(float),
            result.loc[mask, 'rebound_target_position'].astype(float),
        )
        result.loc[mask, 'release_gate_state'] = True
        result.loc[mask, 'target_semantics_revision'] = 'rebound_state_factor_release_target_lift'
    else:
        result.loc[mask, 'target_semantics_revision'] = 'rebound_state_factor_rebound_target_lift'

    result['current_target_position'] = current_position(result)
    result['no_rebound_position'] = np.where(
        result['is_downtrend_state'].astype(bool),
        result['risk_target_position_original'],
        result['teacher_position'],
    )
    result['target_lift'] = result['current_target_position'] - result['no_rebound_position']
    result['factor_effective_lift'] = result['factor_rebound_state'] & (result['target_lift'] > 0.01)
    result['variant'] = ('release' if release_risk_target else 'rebound') + f'_floor_{int(round(floor * 100)):03d}'
    return result.reset_index(drop=True)

def leakage_metrics(frame: pd.DataFrame) -> dict[str, object]:
    mask = frame['factor_rebound_state'].fillna(False).astype(bool)
    if not {'train_start', 'train_end', 'eval_start', 'eval_end'}.issubset(frame.columns):
        return {'factor_rows': int(mask.sum()), 'outside_train_rows': 0, 'eval_overlap_rows': 0, 'leakage_gate': True}
    dates = pd.to_datetime(frame['datetime']).dt.normalize()
    train_start = pd.to_datetime(frame['train_start'], errors='coerce').dt.normalize()
    train_end = pd.to_datetime(frame['train_end'], errors='coerce').dt.normalize()
    eval_start = pd.to_datetime(frame['eval_start'], errors='coerce').dt.normalize()
    eval_end = pd.to_datetime(frame['eval_end'], errors='coerce').dt.normalize()
    train_ready = (dates >= train_start) & (dates <= train_end)
    eval_overlap = (dates >= eval_start) & (dates <= eval_end)
    outside_train_rows = int((mask & ~train_ready.fillna(False)).sum())
    eval_overlap_rows = int((mask & eval_overlap.fillna(False)).sum())
    return {
        'factor_rows': int(mask.sum()),
        'outside_train_rows': outside_train_rows,
        'eval_overlap_rows': eval_overlap_rows,
        'leakage_gate': bool(outside_train_rows == 0 and eval_overlap_rows == 0),
    }

label_config = BinaryLabelDiscoveryConfig(
    min_positive_rows=5,
    max_positive_rate=0.10,
    min_fold_coverage=2,
    min_fold_positive_rows=1,
    min_forward_return_lift=0.0,
    min_target_lift=0.01,
    max_label_conflict_rate=0.0,
)
consensus_config = StateHeadConsensusTargetConfig(min_sources=1, min_positive_votes=1, max_negative_votes=0)
strict_consensus_config = StateHeadConsensusTargetConfig(min_sources=2, min_positive_votes=1, max_negative_votes=0)

variant_frames: dict[str, pd.DataFrame] = {}
variant_rows: list[dict[str, object]] = []
label_scores: list[pd.DataFrame] = []
label_fold_metrics: list[pd.DataFrame] = []
audit_frames: list[pd.DataFrame] = []
lift_group_frames: list[pd.DataFrame] = []
consensus_target_frames: list[pd.DataFrame] = []
consensus_group_frames: list[pd.DataFrame] = []

for release_risk_target in (False, True):
    for floor in (0.35, 0.45, 0.55, 0.65):
        frame = make_variant(floor=floor, release_risk_target=release_risk_target)
        variant = str(frame['variant'].iloc[0])
        variant_frames[variant] = frame

        audit = audit_state_objective_target_lift(
            frame,
            target_position_priority='risk',
            baseline_risk_position_col='risk_target_position_original',
        )
        audit['variant'] = variant
        audit_frames.append(audit)
        audit_summary = summarize_state_objective_target_lift_audit(audit)

        label_result = score_binary_label_candidates(
            frame,
            {variant: frame['factor_rebound_state']},
            group_col='date_key',
            fold_col='fold',
            forward_return_cols=['forward_return_5', 'forward_return_10', 'forward_return_21'],
            target_lift_col='target_lift',
            config=label_config,
        )
        score = label_result.candidate_scores.copy()
        score['variant'] = variant
        label_scores.append(score)
        folds = label_result.fold_metrics.copy()
        folds['variant'] = variant
        label_fold_metrics.append(folds)

        state_rows = frame[[
            'datetime', 'date_key', 'fold', 'factor_rebound_state', 'target_lift',
            'no_rebound_position', 'current_target_position'
        ]].rename(columns={
            'factor_rebound_state': 'label',
            'no_rebound_position': 'default_target_position',
            'current_target_position': 'proposed_target',
        })
        state_rows['predicted_state'] = state_rows['label']
        lift_result = score_state_head_lift_stability(
            state_rows,
            group_col='date_key',
            label_col='label',
            predicted_col='predicted_state',
            target_lift_col='target_lift',
        )
        lift_groups = lift_result.group_metrics.copy()
        lift_groups['variant'] = variant
        lift_group_frames.append(lift_groups)

        consensus = derive_state_head_consensus_targets(
            state_rows,
            group_col='date_key',
            label_col='label',
            default_target_col='default_target_position',
            proposed_target_col='proposed_target',
            target_lift_col='target_lift',
            predicted_col='predicted_state',
            config=consensus_config,
        )
        strict_consensus = derive_state_head_consensus_targets(
            state_rows,
            group_col='date_key',
            label_col='label',
            default_target_col='default_target_position',
            proposed_target_col='proposed_target',
            target_lift_col='target_lift',
            predicted_col='predicted_state',
            config=strict_consensus_config,
        )
        consensus_targets = consensus.consensus_targets.copy()
        consensus_targets['variant'] = variant
        consensus_target_frames.append(consensus_targets)
        consensus_groups = consensus.group_metrics.copy()
        consensus_groups['variant'] = variant
        consensus_group_frames.append(consensus_groups)

        label_best = score.iloc[0].to_dict()
        factor_mask = frame['factor_rebound_state'].fillna(False).astype(bool)
        factor_dates = frame.loc[factor_mask, 'date_key']
        factor_effective = frame.loc[factor_mask, 'factor_effective_lift'].fillna(False).astype(bool)
        leakage = leakage_metrics(frame)
        variant_rows.append({
            'variant': variant,
            'floor_position': float(floor),
            'release_risk_target': bool(release_risk_target),
            **leakage,
            'factor_unique_dates': int(factor_dates.nunique()),
            'factor_years': int(factor_dates.dt.year.nunique()),
            'factor_current_positive_lift_rows': int(factor_effective.sum()),
            'factor_current_positive_lift_rate': float(factor_effective.mean()) if len(factor_effective) else 0.0,
            'audit_decision': str(audit_summary.get('decision')),
            'audit_training_allowed': bool(audit_summary.get('training_allowed', False)),
            'audit_blocked_groups': int(audit_summary.get('blocked_groups', 0)),
            'audit_rebound_rows': int(audit_summary.get('rebound_rows', 0)),
            'audit_current_positive_rebound_lift_rows': int(audit_summary.get('current_positive_rebound_lift_rows', 0)),
            'label_gate': bool(label_best.get('label_gate', False)),
            'label_conflict_rate': float(label_best.get('label_conflict_rate', np.nan)),
            'positive_rate': float(label_best.get('positive_rate', np.nan)),
            'forward_return_5_lift': float(label_best.get('forward_return_5_lift', np.nan)),
            'forward_return_10_lift': float(label_best.get('forward_return_10_lift', np.nan)),
            'forward_return_21_lift': float(label_best.get('forward_return_21_lift', np.nan)),
            'target_lift_mean': float(label_best.get('target_lift_mean', np.nan)),
            'lift_stability_gate': bool(lift_result.summary.get('stability_gate', False)),
            'unstable_effective_lift_rows': int(lift_result.summary.get('unstable_effective_lift_rows', 0)),
            'positive_effective_lift_rows': int(lift_result.summary.get('positive_effective_lift_rows', 0)),
            'consensus_gate': bool(consensus.summary.get('consensus_gate', False)),
            'consensus_target_groups': int(consensus.summary.get('consensus_target_groups', 0)),
            'strict_consensus_target_groups': int(strict_consensus.summary.get('consensus_target_groups', 0)),
        })

variant_metrics = pd.DataFrame(variant_rows)
variant_metrics['selection_gate'] = (
    variant_metrics['release_risk_target'].astype(bool)
    & variant_metrics['leakage_gate'].astype(bool)
    & variant_metrics['label_gate'].astype(bool)
    & variant_metrics['lift_stability_gate'].astype(bool)
    & variant_metrics['consensus_gate'].astype(bool)
    & (variant_metrics['strict_consensus_target_groups'] > 0)
    & (variant_metrics['factor_rows'] > 0)
    & (variant_metrics['factor_current_positive_lift_rows'] == variant_metrics['factor_rows'])
    & (variant_metrics['factor_unique_dates'] >= 5)
    & (variant_metrics['factor_years'] >= 3)
)
eligible = variant_metrics[variant_metrics['selection_gate']].sort_values(['floor_position', 'variant'])
selected_variant = str(eligible.iloc[0]['variant']) if not eligible.empty else ''
selected_metrics = eligible.iloc[0].to_dict() if selected_variant else {}
selected_contract = variant_frames[selected_variant].copy() if selected_variant else pd.DataFrame()
selected_factor_targets = selected_contract[selected_contract['factor_rebound_state']].copy() if selected_variant else pd.DataFrame()

all_label_scores = pd.concat(label_scores, ignore_index=True) if label_scores else pd.DataFrame()
all_label_fold_metrics = pd.concat(label_fold_metrics, ignore_index=True) if label_fold_metrics else pd.DataFrame()
all_audit = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
all_lift_groups = pd.concat(lift_group_frames, ignore_index=True) if lift_group_frames else pd.DataFrame()
all_consensus_targets = pd.concat(consensus_target_frames, ignore_index=True) if consensus_target_frames else pd.DataFrame()
all_consensus_groups = pd.concat(consensus_group_frames, ignore_index=True) if consensus_group_frames else pd.DataFrame()

variant_metrics.to_csv(CARD_DIR / 'target_contract_variants.csv', index=False)
all_label_scores.to_csv(CARD_DIR / 'label_stability_scores.csv', index=False)
all_label_fold_metrics.to_csv(CARD_DIR / 'label_fold_metrics.csv', index=False)
all_audit.to_csv(CARD_DIR / 'target_lift_audit.csv', index=False)
all_lift_groups.to_csv(CARD_DIR / 'lift_stability_group_metrics.csv', index=False)
all_consensus_targets.to_csv(CARD_DIR / 'consensus_targets.csv', index=False)
all_consensus_groups.to_csv(CARD_DIR / 'consensus_group_metrics.csv', index=False)
if selected_variant:
    selected_contract.to_csv(CARD_DIR / 'selected_target_contract.csv', index=False)
    selected_factor_targets.to_csv(CARD_DIR / 'selected_factor_target_rows.csv', index=False)

decision = (
    'rebound_state_factor_target_contract_ready_for_label_stability_no_training'
    if selected_variant
    else 'rebound_state_factor_target_contract_blocked_no_training'
)
next_mechanism = (
    'rerun_state_head_label_stability_or_bc_replay_with_rebound_state_factor_target_contract'
    if selected_variant
    else 'mine_alternative_rebound_state_factor_or_reduce_target_floor'
)
selection = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selected_variant': selected_variant,
    'selected_candidate': str(factor_summary.get('selected_candidate') or handoff_summary.get('selected_candidate') or ''),
    'selected_floor_position': float(selected_metrics.get('floor_position', np.nan)) if selected_metrics else None,
    'training_allowed': False,
    'audit_training_allowed': bool(selected_metrics.get('audit_training_allowed', False)) if selected_metrics else False,
    'target_contract_gate': bool(selected_variant),
    'bc_readiness_gate': False,
    'next_training_allowed_after_target_revision': bool(selected_variant),
    'next_mechanism': next_mechanism,
    'source_card': 'rebound_state_factor_target_contract_handoff_v1',
    'source_target_experiment': 'state_objective_fold_native_teacher_router_v1',
    'source_factor_experiment': 'rebound_state_factor_mining_v1',
    'selection_reason': 'selected smallest release-style floor whose factor rows all produce effective risk-priority target lift with zero duplicate-date label conflict',
}
summary = {
    **selection,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'source_target_path': str(TARGET_PATH.relative_to(ROOT)),
    'source_factor_rows_path': str(FACTOR_ROWS_PATH.relative_to(ROOT)),
    'target_rows': int(len(targets)),
    'target_unique_dates': int(targets['date_key'].nunique()),
    'factor_full_history_rows': int(len(factor_rows)),
    'factor_full_history_unique_dates': int(factor_rows['date_key'].nunique()),
    'target_overlap_rows': int(targets['date_key'].isin(selected_dates).sum()),
    'target_overlap_unique_dates': int(targets.loc[targets['date_key'].isin(selected_dates), 'date_key'].nunique()),
    'target_overlap_years': int(targets.loc[targets['date_key'].isin(selected_dates), 'date_key'].dt.year.nunique()),
    'selected_metrics': json_safe(selected_metrics),
    'variant_count': int(len(variant_metrics)),
    'eligible_variant_count': int(len(eligible)),
    'target_lift_summary': json_safe(selected_metrics),
    'factor_summary': json_safe({
        key: factor_summary.get(key)
        for key in [
            'selected_candidate', 'selected_forward_return_5_lift', 'selected_forward_return_10_lift',
            'selected_forward_return_21_lift', 'selected_target_lift_mean', 'selected_hold_5_max_drawdown',
            'selected_full_years', 'selected_full_rows'
        ]
        if key in factor_summary
    }),
}
selection_payload = {'selection': selection, 'summary': summary}
(CARD_DIR / 'selection.json').write_text(json.dumps(json_safe(selection_payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')
(CARD_DIR / 'summary.json').write_text(json.dumps(json_safe(summary), indent=2, sort_keys=True) + '\n', encoding='utf-8')
(CARD_DIR / 'target_lift_summary.json').write_text(json.dumps(json_safe({'selected_variant': selected_variant, 'selected_metrics': selected_metrics}), indent=2, sort_keys=True) + '\n', encoding='utf-8')

print(json.dumps(json_safe(summary), indent=2, sort_keys=True))


{
  "audit_training_allowed": true,
  "bc_readiness_gate": false,
  "decision": "rebound_state_factor_target_contract_ready_for_label_stability_no_training",
  "eligible_variant_count": 4,
  "experiment": "rebound_state_factor_target_contract_v1",
  "factor_full_history_rows": 34,
  "factor_full_history_unique_dates": 34,
  "factor_summary": {
    "selected_candidate": "spread_confirmed_pullback__mom5_leq0.20_ratio21_geq0.75",
    "selected_forward_return_10_lift": 0.051703347429554,
    "selected_forward_return_21_lift": 0.23378623281615393,
    "selected_forward_return_5_lift": 0.11752518859252593,
    "selected_full_rows": 34,
    "selected_full_years": 13,
    "selected_hold_5_max_drawdown": -0.15896233250465586,
    "selected_target_lift_mean": 0.0
  },
  "generated_at": "2026-06-14T11:59:23.425273+00:00",
  "next_mechanism": "rerun_state_head_label_stability_or_bc_replay_with_rebound_state_factor_target_contract",
  "next_training_allowed_after_target_revision": true,
  "selected

In [2]:
from IPython.display import Markdown, display

if selected_variant:
    conclusion = f"""
## Conclusion

`{selected_variant}` is the selected target contract. It is a release-style semantic revision: factor-matched rebound rows lift `rebound_target_position` and also lift `risk_target_position` so the risk-priority target selector does not erase the signal during downtrend states.

Key evidence:
- Factor target rows: `{int(selected_metrics['factor_rows'])}` across `{int(selected_metrics['factor_unique_dates'])}` unique dates and `{int(selected_metrics['factor_years'])}` years.
- Eval leakage rows: `{int(selected_metrics['eval_overlap_rows'])}`; outside-train rows: `{int(selected_metrics['outside_train_rows'])}`.
- Duplicate-date label conflict rate: `{selected_metrics['label_conflict_rate']:.4f}`.
- Factor rows with effective target lift: `{int(selected_metrics['factor_current_positive_lift_rows'])}/{int(selected_metrics['factor_rows'])}`.
- 5d/10d/21d forward return lift: `{selected_metrics['forward_return_5_lift']:.4f}`, `{selected_metrics['forward_return_10_lift']:.4f}`, `{selected_metrics['forward_return_21_lift']:.4f}`.
- Consensus target groups: `{int(selected_metrics['consensus_target_groups'])}`; strict two-source consensus groups: `{int(selected_metrics['strict_consensus_target_groups'])}`.

Decision: `{decision}`. Training remains disabled in this notebook. Next action: `{next_mechanism}`.
"""
else:
    conclusion = f"""
## Conclusion

No target contract variant passed the selection gates. Training remains disabled. Next action: `{next_mechanism}`.
"""

display(Markdown(conclusion))



## Conclusion

`release_floor_035` is the selected target contract. It is a release-style semantic revision: factor-matched rebound rows lift `rebound_target_position` and also lift `risk_target_position` so the risk-priority target selector does not erase the signal during downtrend states.

Key evidence:
- Factor target rows: `20` across `11` unique dates and `3` years.
- Eval leakage rows: `0`; outside-train rows: `0`.
- Duplicate-date label conflict rate: `0.0000`.
- Factor rows with effective target lift: `20/20`.
- 5d/10d/21d forward return lift: `0.1178`, `0.0528`, `0.2359`.
- Consensus target groups: `11`; strict two-source consensus groups: `9`.

Decision: `rebound_state_factor_target_contract_ready_for_label_stability_no_training`. Training remains disabled in this notebook. Next action: `rerun_state_head_label_stability_or_bc_replay_with_rebound_state_factor_target_contract`.
